In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1806, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 1806 (delta 148), reused 0 (delta 0), pack-reused 1613 (from 2)
Receiving objects: 100% (1806/1806), 629.18 MiB | 62.30 MiB/s, done.
Resolving deltas: 100% (1151/1151), done.
Updating files: 100% (233/233), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.1 MB/s eta 0:00:00


In [3]:
# ! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

In [4]:
! cd /kaggle/working/RecSys_challenge_2025

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [6]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [7]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [8]:
db_name = 'multvae_torch_study.db'
json_name = 'multvae_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '20',
    'n_trials': 15
}

In [9]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [10]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [11]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [12]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [13]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

import torch
from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [14]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [15]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

Error reading data from table 'studies': no such table: studies


In [16]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_multvae(optuna_trial):
    early_stopping_params = {
        "validation_every_n": 1,
        "stop_on_validation": True,
        "evaluator_object": evaluator,
        "lower_validations_allowed": 15,
        "validation_metric": "RECALL",
    }
    
    n_layers = optuna_trial.suggest_categorical('n_layers', [1, 2])
    layer_1 = optuna_trial.suggest_int('layer_1', 64, 600, log=True)
    p_dims = [layer_1]
    if n_layers == 2:
        layer_2 = optuna_trial.suggest_int('layer_2', layer_1, 2000, log=True)
        p_dims.append(layer_2)
    
    p_dims.append(URM_train.shape[1])
    
    learning_rate = optuna_trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    l2_reg = optuna_trial.suggest_float('l2_reg', 1e-6, 1e-2, log=True)
    dropout = optuna_trial.suggest_float('dropout', 0.0, 0.5)
    batch_size = optuna_trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512, 1024])
    anneal_cap = optuna_trial.suggest_float('anneal_cap', 0.05, 1.0)
    total_anneal_steps = optuna_trial.suggest_int('total_anneal_steps', 500, 300000, log=True)
    
    recommender_instance = MultVAERecommender_PyTorch(URM_train, use_gpu=True)
    
    recommender_instance.fit(
        epochs=500,
        learning_rate=learning_rate,
        l2_reg=l2_reg,
        dropout=dropout,
        batch_size=batch_size,
        anneal_cap=anneal_cap,
        total_anneal_steps=total_anneal_steps,
        p_dims=p_dims,
        sgd_mode='adam',
        **early_stopping_params
    )
    
    epochs_trained = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    optuna_trial.set_user_attr("epochs", epochs_trained)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [17]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='multvae_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_multvae, n_trials=config['n_trials'])

[I 2026-01-04 10:52:46,587] A new study created in RDB with name: multvae_Optimization


Previous trials found: 0


/kaggle/working/RecSys_challenge_2025/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:307: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(


MultVAERecommender_PyTorch: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 12.56 sec. Users per second: 2154
MultVAERecommender_PyTorch: CUTOFF: 20 - PRECISION: 0.0334522, PRECISION_RECALL_MIN_DEN: 0.0458000, RECALL: 0.0345670, MAP: 0.0099447, MAP_MIN_DEN: 0.0130776, MRR: 0.1353111, NDCG: 0.0454703, F1: 0.0340004, HIT_RATE: 0.4352163, ARHR_ALL_HITS: 0.1623591, NOVELTY: 0.0295899, AVERAGE_POPULARITY: 0.2450782, DIVERSITY_MEAN_INTER_LIST: 0.5390688, DIVERSITY_HERFINDAHL: 0.9769524, COVERAGE_ITEM: 0.0467786, COVERAGE_ITEM_HIT: 0.0172191, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.4347666, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0064614, SHANNON_ENTROPY: 5.8973751, RATIO_DIVERSITY_HERFINDAHL: 0.9777086, RATIO_DIVERSITY_GINI: 0.0238911, RATIO_SHANNON_ENTROPY: 0.5259994, RATIO_AVERAGE_POPULARITY: 1.1531182, RATIO_NOVELTY: 0.2047058, 

MultVAERecommender_PyTorch: New best model found! Updating.
MultVAERecommender_PyTorch: Epoch 1 of 500. Elaps

[I 2026-01-04 11:48:49,230] Trial 0 finished with value: 0.23616487715032503 and parameters: {'n_layers': 1, 'layer_1': 64, 'learning_rate': 0.0002834223042206766, 'l2_reg': 0.008497081572287905, 'dropout': 0.06698869907080357, 'batch_size': 1024, 'anneal_cap': 0.4127775633380547, 'total_anneal_steps': 155969}. Best is trial 0 with value: 0.23616487715032503.


...

MultVAERecommender_PyTorch: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 12.45 sec. Users per second: 2175
MultVAERecommender_PyTorch: CUTOFF: 20 - PRECISION: 0.0957162, PRECISION_RECALL_MIN_DEN: 0.1429151, RECALL: 0.1159193, MAP: 0.0353659, MAP_MIN_DEN: 0.0505197, MRR: 0.3181353, NDCG: 0.1372446, F1: 0.1048534, HIT_RATE: 0.7694979, ARHR_ALL_HITS: 0.4592418, NOVELTY: 0.0246269, AVERAGE_POPULARITY: 0.6125733, DIVERSITY_MEAN_INTER_LIST: 0.3775993, DIVERSITY_HERFINDAHL: 0.9688793, COVERAGE_ITEM: 0.0143493, COVERAGE_ITEM_HIT: 0.0111924, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.7687027, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0043307, SHANNON_ENTROPY: 5.1635771, RATIO_DIVERSITY_HERFINDAHL: 0.9696292, RATIO_DIVERSITY_GINI: 0.0160127, RATIO_SHANNON_ENTROPY: 0.4605504, RATIO_AVERAGE_POPULARITY: 2.8822208, RATIO_NOVELTY: 0.1703710, 

MultVAERecommender_PyTorch: New best model found! Updating.
MultVAERecommender_PyTorch: Epoch 1 of 500. Elaps

[I 2026-01-04 18:31:26,112] Trial 13 finished with value: 0.24878090342135772 and parameters: {'n_layers': 1, 'layer_1': 220, 'learning_rate': 0.0008135674080793447, 'l2_reg': 0.00018175251768922548, 'dropout': 0.32003460548414503, 'batch_size': 256, 'anneal_cap': 0.4996804224986612, 'total_anneal_steps': 29847}. Best is trial 13 with value: 0.24878090342135772.


MultVAERecommender_PyTorch: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 12.76 sec. Users per second: 2122
MultVAERecommender_PyTorch: CUTOFF: 20 - PRECISION: 0.0894355, PRECISION_RECALL_MIN_DEN: 0.1340284, RECALL: 0.1090613, MAP: 0.0322526, MAP_MIN_DEN: 0.0463169, MRR: 0.2953584, NDCG: 0.1277986, F1: 0.0982782, HIT_RATE: 0.7488824, ARHR_ALL_HITS: 0.4237863, NOVELTY: 0.0246026, AVERAGE_POPULARITY: 0.6152528, DIVERSITY_MEAN_INTER_LIST: 0.3351372, DIVERSITY_HERFINDAHL: 0.9667562, COVERAGE_ITEM: 0.0106185, COVERAGE_ITEM_HIT: 0.0087530, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.7481085, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0041079, SHANNON_ENTROPY: 5.0327387, RATIO_DIVERSITY_HERFINDAHL: 0.9675045, RATIO_DIVERSITY_GINI: 0.0151889, RATIO_SHANNON_ENTROPY: 0.4488806, RATIO_AVERAGE_POPULARITY: 2.8948278, RATIO_NOVELTY: 0.1702026, 

MultVAERecommender_PyTorch: New best model found! Updating.
MultVAERecommender_PyTorch: Epoch 1 of 500. Elaps

[I 2026-01-04 19:24:17,235] Trial 14 finished with value: 0.2416275305365199 and parameters: {'n_layers': 2, 'layer_1': 211, 'layer_2': 1863, 'learning_rate': 8.60095859299934e-05, 'l2_reg': 0.0002752444631276462, 'dropout': 0.1418832801761874, 'batch_size': 256, 'anneal_cap': 0.47573128282431887, 'total_anneal_steps': 605}. Best is trial 13 with value: 0.24878090342135772.


CPU times: user 8h 18min 45s, sys: 12min 57s, total: 8h 31min 43s
Wall time: 8h 31min 33s


In [18]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            best_params['epochs'] = study.best_trial.user_attrs['epochs']
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.248781 <= Saved: 0.253126
